In [ ]:
import pandapower as pp
import pyomo.environ as pe
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import pickle
import simbench
import json
import os
import sys
import warnings

# Add Model_Build_Manifest to system path
sys.path.insert(0, './Model_Build_Manifest')

# Import framework components
from simulator import ManifestFactory, ModelData, CONSTRAINT_LIBRARY
from controller import ModelAssembler, MPCController, SolverConfig

warnings.filterwarnings('ignore')

In [ ]:
class DecoupledSimulator:
    """
    Physics Simulator: Manages grid state, time-series data, and storage SOC.
    Agnostic to the optimizer; structure exposed via Manifest.
    """
    def __init__(self, Enet, config_path="./data/elec_simbench_config.json"):
        self.Enet_original = Enet

        # Load config
        with open(config_path, 'r') as f:
            self.config = json.load(f)

        # Init physical model copy
        self.Enet = pp.pandapowerNet(self.Enet_original)

        # Init storage SOC (kWh) from pandapower object
        self.storage_soc = {}
        if hasattr(self.Enet, 'storage') and not self.Enet.storage.empty:
            for idx in self.Enet.storage.index:
                soc_pct = self.Enet.storage.at[idx, 'soc_percent']
                max_e = self.Enet.storage.at[idx, 'max_e_mwh']
                self.storage_soc[idx] = (soc_pct / 100.0) * max_e

        # Cache Time Series profiles
        self.profiles = {
            'load_p': self.Enet_original.get('profiles_load_p'),
            'load_q': self.Enet_original.get('profiles_load_q'),
            'pv_p': self.Enet_original.get('profiles_pv_p')
        }

        self.current_step = 0

    def generate_manifest(self):
        """Generate Manifest. The only structural exposure to the outside."""
        print("Simulator: Generating Manifest...")
        model_data = ModelData(self.Enet, self.config)
        factory = ManifestFactory(model_data)
        return factory.create_manifest()

    def step_forward(self, step, prefix=0):
        """Advance time step; update boundary conditions (Load/PV)."""
        self.current_step = step

        # 1. Update Load (P/Q)
        if self.profiles['load_p'] is not None and len(self.profiles['load_p']) > prefix:
            for i, idx in enumerate(self.Enet.load.index):
                if i < len(self.profiles['load_p'][prefix]):
                    self.Enet.load.at[idx, 'p_mw'] = self.profiles['load_p'][prefix][i][step]

        if self.profiles['load_q'] is not None and len(self.profiles['load_q']) > prefix:
             for i, idx in enumerate(self.Enet.load.index):
                if i < len(self.profiles['load_q'][prefix]):
                    self.Enet.load.at[idx, 'q_mvar'] = self.profiles['load_q'][prefix][i][step]

        # 2. Update PV (Sgen P)
        if self.profiles['pv_p'] is not None and len(self.profiles['pv_p']) > prefix:
            for i, idx in enumerate(self.Enet.sgen.index):
                if i < len(self.profiles['pv_p'][prefix]):
                    self.Enet.sgen.at[idx, 'p_mw'] = self.profiles['pv_p'][prefix][i][step]

        # 3. Update Storage SOC % (Physical state sync only)
        if hasattr(self.Enet, 'storage'):
            for idx in self.Enet.storage.index:
                if idx in self.storage_soc:
                    max_e = self.Enet.storage.at[idx, 'max_e_mwh']
                    if max_e > 0:
                        self.Enet.storage.at[idx, 'soc_percent'] = (self.storage_soc[idx] / max_e) * 100.0

    def get_runtime_state(self):
        """
        Extract current runtime state dict.
        Contains pure data, no Pandapower objects.
        """
        state = {
            'step': self.current_step,
            'loads': {},
            'sgens': {},
            'gens': {},
            'storage': {}
        }

        # Extract Loads
        for idx in self.Enet.load.index:
            state['loads'][int(idx)] = {
                'p_mw': float(self.Enet.load.at[idx, 'p_mw']),
                'q_mvar': float(self.Enet.load.at[idx, 'q_mvar'])
            }

        # Extract PV/Sgen
        for idx in self.Enet.sgen.index:
            state['sgens'][int(idx)] = {
                'p_mw': float(self.Enet.sgen.at[idx, 'p_mw']),
                'q_mvar': float(self.Enet.sgen.at[idx, 'q_mvar'])
            }

        # Extract Gen
        for idx in self.Enet.gen.index:
             state['gens'][int(idx)] = {
                'p_mw': float(self.Enet.gen.at[idx, 'p_mw']),
                'q_mvar': float(self.Enet.gen.at[idx, 'q_mvar'])
            }

        return state

    def apply_control_signals(self, signals, dt_min=15):
        """
        Update physical state based on controller signals.
        Update logic: E(t+1) = E(t) + P(t) * dt
        """
        if not signals or 'storage_p' not in signals:
            return

        dt_hours = dt_min / 60.0

        for idx_str, p_val in signals['storage_p'].items():
            idx = int(idx_str)
            if idx in self.storage_soc:
                # Update energy (Assuming P > 0 is charging)
                self.storage_soc[idx] += p_val * dt_hours

                # Clip to physical limits
                max_e = self.Enet.storage.at[idx, 'max_e_mwh']
                min_e = self.Enet.storage.at[idx, 'min_e_mwh'] if 'min_e_mwh' in self.Enet.storage else 0
                self.storage_soc[idx] = np.clip(self.storage_soc[idx], min_e, max_e)

In [ ]:
class DecoupledController:
    """
    Decoupled Controller: Builds model from Manifest and solves using runtime data.
    Independent of Pandapower.
    """
    def __init__(self, config):
        self.config = config
        self.assembler = ModelAssembler(CONSTRAINT_LIBRARY)
        self.mpc = None
        self.model = None

    def initialize(self, manifest):
        """Build optimization model from Manifest."""
        print("Controller: Receiving manifest and building Pyomo model...")
        self.assembler.register_manifest(manifest)

        # Configure Solver
        raw_options = self.config.get('solver_options', {})
        ipopt_options = {k: v for k, v in raw_options.items() if k not in ['verbose', 'tee']}
        if 'tol' not in ipopt_options: ipopt_options['tol'] = 1e-8

        solver_cfg = SolverConfig(
            solver_name=self.config.get('solver', 'ipopt'),
            options=ipopt_options,
            verbose=raw_options.get('verbose', True)
        )

        # Init MPC and build model
        self.mpc = MPCController(self.assembler, self.config, solver_cfg)
        self.model = self.mpc.build_model()

        # Set objective and fix slack nodes
        self._setup_objective(manifest)
        self.mpc.fix_slack_nodes()

    def _setup_objective(self, manifest):
        """Set objective function (minimize slack exchange)."""
        objective_type = self.config.get('objective', 'quadratic_exchange')

        if objective_type == 'quadratic_exchange':
            slack_nodes = manifest['Sets']['slack_nodes']['data']

            # Objective: sum(P_slack^2 + Q_slack^2)
            expr = sum(
                self.model.P[i, t] ** 2 + self.model.Q[i, t] ** 2
                for i in slack_nodes
                for t in self.model.T
            )
            self.model.obj = pe.Objective(expr=expr, sense=pe.minimize)
            print("Controller: Objective function set to 'quadratic_exchange'")
        else:
            print(f"Controller: Warning - Objective type '{objective_type}' not implemented.")

    def solve_step(self, runtime_state):
        """Inject state -> Solve -> Return control signals."""
        if self.model is None:
            raise RuntimeError("Model not initialized!")

        # 1. Inject physical state (Fix variables)
        self._inject_runtime_data(runtime_state)

        # 2. Solve
        try:
            results = self.mpc.solve()

            # 3. Extract control signals (Storage Power)
            signals = {'storage_p': {}}
            if hasattr(self.model, 'Psto'):
                for idx in self.model.storage_nodes:
                    # Get result at t=0
                    signals['storage_p'][str(idx)] = pe.value(self.model.Psto[idx, 0])

            return signals, results

        except Exception as e:
            print(f"Controller Error: {e}")
            return None, None

    def _inject_runtime_data(self, state):
        """Helper: Fix Load/Gen/SGen power values."""
        # Fix Loads
        if 'loads' in state:
            for idx, vals in state['loads'].items():
                if idx in self.model.pq_nodes:
                    self.model.Pload[idx, :].fix(vals['p_mw'])
                    self.model.Qload[idx, :].fix(vals['q_mvar'])

        # Fix SGens (PV)
        if 'sgens' in state:
            for idx, vals in state['sgens'].items():
                if idx in self.model.sgen_nodes:
                    self.model.Psgen[idx, :].fix(vals['p_mw'])
                    self.model.Qsgen[idx, :].fix(vals['q_mvar'])

        # Fix Gens
        if 'gens' in state:
            for idx, vals in state['gens'].items():
                if idx in self.model.generator_nodes:
                    self.model.Pgen[idx, :].fix(vals['p_mw'])
                    self.model.Qgen[idx, :].fix(vals['q_mvar'])

    def extract_full_results(self, Enet_ref):
        """Convert raw library results to DataFrame."""
        if self.mpc and self.model:
            raw_results = self.mpc.extract_results(Enet_ref)

            # Assume Horizon=1, take column 0
            # 1. Bus results
            res_bus = pd.DataFrame({
                'vm_pu': raw_results['V_pu'][:, 0],
                'va_degree': np.degrees(raw_results['Theta_rad'][:, 0]),
                'p_mw': raw_results['P_mw'][:, 0],
                'q_mvar': raw_results['Q_mvar'][:, 0]
            })

            # 2. Line results
            res_line = pd.DataFrame({
                'p_from_mw': raw_results['pf_mw'][:, 0],
                'q_from_mvar': raw_results['qf_mvar'][:, 0],\
                'p_to_mw': raw_results['pt_mw'][:, 0],
                'q_to_mvar': raw_results['qt_mvar'][:, 0]
            })

            # 3. Objective value
            res_obj = raw_results['objective']

            return res_bus, res_line, res_obj

        return None, None, None

In [ ]:
# 1. Load SimBench Network
code = '1-LV-rural1--1-sw'
Enet = simbench.get_simbench_net(code)

# 2. Modify Network: Add PVs
Enet.sgen = pd.concat([Enet.sgen, Enet.sgen[0:4]], ignore_index=True)

# Set new PV parameters
new_pv_indices = [8, 9, 10, 11]
target_buses = [12, 14, 6, 9]
pv_names = ["LV1.101 SGen 9", "LV1.101 SGen 10", "LV1.101 SGen 11", "LV1.101 SGen 12"]

for i, idx in enumerate(new_pv_indices):
    Enet.sgen.at[idx, "name"] = pv_names[i]
    Enet.sgen.at[idx, "bus"] = target_buses[i]

# 3. Set Storage Constraints
Enet.storage["max_p_mw"] = -Enet.storage["min_p_mw"]
Enet.ext_grid['vm_pu'] = 1

# 4. Generate Scenarios (Load/PV Profiles)
def is_index_affected(index, affected_idx):
    if isinstance(affected_idx, int): return index == affected_idx
    elif isinstance(affected_idx, (list, np.ndarray)): return index in affected_idx
    return False

# Init data structures
profiles_load_p = [[], []] # [Scenario0, Scenario1]
profiles_load_q = [[], []]
profiles_pv_p = [[], []]

# Params
TIME_SLICE = slice(96*5, 96*7)
T_START_GD = 48
DURATION = 8
GD_FACTOR = 1.5
GD_AFFECTED_PV_IDX = [0, 8, 9, 10, 7]

# Build Load Profiles
for i, profile in enumerate(Enet.load.profile.values):
    p0_p = Enet.profiles["load"][profile + "_pload"].values[TIME_SLICE] * Enet.load.at[i, "p_mw"]
    p0_q = Enet.profiles["load"][profile + "_qload"].values[TIME_SLICE] * Enet.load.at[i, "q_mvar"]

    # S0 (Base) & S2 (Same load)
    profiles_load_p[0].append(p0_p.copy())
    profiles_load_q[0].append(p0_q.copy())
    profiles_load_p[1].append(p0_p.copy())
    profiles_load_q[1].append(p0_q.copy())

# Build PV Profiles (with disturbance)
for i, profile in enumerate(Enet.sgen.profile.values):
    p0_pv = Enet.profiles["renewables"][profile].values[TIME_SLICE] * Enet.sgen.at[i, "p_mw"]

    # S0: Original
    profiles_pv_p[0].append(p0_pv.copy())

    # S2: Disturbed
    p2_pv = p0_pv.copy()
    if is_index_affected(i, GD_AFFECTED_PV_IDX):
        p2_pv[T_START_GD : T_START_GD + DURATION] = GD_FACTOR * p2_pv[T_START_GD : T_START_GD + DURATION]
    profiles_pv_p[1].append(p2_pv)

# Attach profiles to Enet (Data carrier)
Enet["profiles_load_p"] = np.array(profiles_load_p)
Enet["profiles_load_q"] = np.array(profiles_load_q)
Enet["profiles_pv_p"] = np.array(profiles_pv_p)

# Rounding
for key, df in Enet.items():
    if isinstance(df, pd.DataFrame):
        Enet[key] = df.round(4)

# Backup Enet
Enet_backup = pp.pandapowerNet(Enet)
print("Data preparation complete.")

# -------------------------------------------------------------------------
# Simulation Configuration
# -------------------------------------------------------------------------
# Select Scenario: S2 (PtG + Disturbance)
scenario_path = f"./results/S3/decoupled_simulation_{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
os.makedirs(scenario_path, exist_ok=True)

# 1. Prepare Environment (Restore backup)
Enet_sim = pp.pandapowerNet(Enet_backup)
Enet_sim.line.at[2, "max_i_ka"] = 0.0404

# 2. Add PtG (Power-to-Gas) components
# Must be done before Simulator init for Manifest generation
KopplungPtG = {
    "max_p_mw": {0: 0.05},
    "bus": {0: 11},
    "junction": {0: 2},
    "heating_value": {0: 11.55},
    "efficiency": {0: 0.9}
}
Enet_sim["PtG"] = pd.DataFrame(data=KopplungPtG)

# Config for Controller
sim_config = {
  "solver": "ipopt",
  "solver_options": {
    "verbose": True,
    "tee": False
  },
  "with_timeseries": False,
  "mpc_with_initial_values": False,
  "mpc_steps": 96,
  "horizon": 3,
  "dt_min": 15,
  "flow_constraint": False,
  "v_min": 0.9,
  "v_max": 1.1,
  "with_storage_ramp": False,
  "objective": "quadratic_exchange"
}

# -------------------------------------------------------------------------
# Instantiate Modules
# -------------------------------------------------------------------------
simulator = DecoupledSimulator(Enet_sim) # Physics Domain
controller = DecoupledController(sim_config) # Control Domain

# -------------------------------------------------------------------------
# Handshake Phase
# -------------------------------------------------------------------------
# Simulator generates Manifest
manifest = simulator.generate_manifest()

# Save manifest (Optional)
with open(f"{scenario_path}/manifest.json", 'w') as f:
    json.dump(manifest, f, indent=2)

# Controller initializes with Manifest
controller.initialize(manifest)

# -------------------------------------------------------------------------
# Orchestration Loop
# -------------------------------------------------------------------------
simulation_steps = 96
results_store = []

print(f"Starting simulation for {simulation_steps} steps...")

for t in range(simulation_steps):
    step_start = datetime.datetime.now()

    # ---------------------------------------------------------------------
    # 1. Physics Evolution (Simulator Domain)
    # ---------------------------------------------------------------------
    # Switch profile (S0=0, S2=1) midway to simulate scenario change
    current_prefix = 0 if t < 48 else 1

    # Advance step, update Load/PV/SOC
    simulator.step_forward(t, prefix=current_prefix)

    # Get runtime state (Data only)
    runtime_state = simulator.get_runtime_state()

    # ---------------------------------------------------------------------
    # 2. Optimization (Controller Domain)
    # ---------------------------------------------------------------------
    # Send state, get control signals
    control_signals, raw_results = controller.solve_step(runtime_state)

    if control_signals is None:
        print(f"Step {t}: Optimization failed.")
        break

    # ---------------------------------------------------------------------
    # 3. Closed-loop Feedback
    # ---------------------------------------------------------------------
    # Apply control signals to physics
    simulator.apply_control_signals(control_signals)

    # ---------------------------------------------------------------------
    # 4. Logging
    # ---------------------------------------------------------------------
    # Extract results
    res_bus, res_line, res_obj = controller.extract_full_results(simulator.Enet)

    # Save step results
    with open(f"{scenario_path}/model_{t}.pkl", 'wb') as f:
        pickle.dump({
            'res_bus': res_bus,
            'res_line': res_line,
            'res_obj_fcn': res_obj,
            'control_signals': control_signals,
            'step': t
        }, f)

    step_end = datetime.datetime.now()
    if t % 10 == 0:
        print(f"Step {t}/{simulation_steps} completed in {(step_end - step_start).total_seconds():.2f}s")

print(f"Simulation complete. Results saved to {scenario_path}")